# 03. Export to Claude Desktop

`papers_scored.csv` 상위 후보를 markdown으로 export합니다.


In [ ]:
import math
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_scored.csv')
RESEARCH_QUESTION = 'How does AI-supported formative feedback influence self-regulated learning in educational technology environments?'
TOP_N = 15
top = df.head(TOP_N).copy()


In [ ]:
def safe_str(value, default=''):
    if value is None:
        return default
    if isinstance(value, float) and math.isnan(value):
        return default
    return str(value).replace('\n', ' ').strip() or default


def safe_int(value, default=0):
    try:
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return default
        return int(value)
    except (TypeError, ValueError):
        return default


def clip(value, n):
    return value if len(value) <= n else value[: n - 3] + '...'


def to_markdown(df):
    lines = [
        '# Educational Technology Paper Candidates',
        '',
        f'Research question: {RESEARCH_QUESTION}',
        '',
        'Use this export with `reference_quality_check(rr)` first. Do not treat ranking as final until full text has been checked.',
        '',
    ]
    header = ['#', 'Role', 'Title', 'Authors', 'Year', 'Citations', 'Cit/yr', 'Topic', 'Score', 'DOI']
    lines += ['| ' + ' | '.join(header) + ' |', '|' + '|'.join(['---'] * len(header)) + '|']
    for i, row in df.reset_index(drop=True).iterrows():
        cells = [
            str(i + 1),
            safe_str(row.get('candidate_role')),
            clip(safe_str(row.get('title')).replace('|', '\\|'), 110),
            clip(safe_str(row.get('authors')).replace('|', '\\|'), 70),
            safe_str(row.get('year')),
            str(safe_int(row.get('cited_by_count'))),
            safe_str(row.get('citations_per_year')),
            safe_str(row.get('topic_relevance_score')),
            safe_str(row.get('quality_score')),
            safe_str(row.get('doi')),
        ]
        lines.append('| ' + ' | '.join(cells) + ' |')
    lines += ['', '## Abstracts and Screening Notes', '']
    for i, row in df.reset_index(drop=True).iterrows():
        lines += [
            f'### [{i + 1}] {safe_str(row.get("title"), "(no title)")}',
            f'- role: {safe_str(row.get("candidate_role"))}',
            f'- keyword_hits: {safe_str(row.get("keyword_hits"), "(none)")}',
            f'- venue: {safe_str(row.get("venue"), "(unknown)")}',
            f'- open_access_url: {safe_str(row.get("oa_url"), "(not found)")}',
            f'- abstract: {safe_str(row.get("abstract"), "(no abstract)")}',
            '',
        ]
    lines += [
        '## Claude Desktop next prompt',
        '',
        '```text',
        'reference_quality_check(rr)을 적용해줘.',
        '',
        f'내 연구 질문: {RESEARCH_QUESTION}',
        '',
        '아래 후보 논문을 관련성, 신뢰성, 방법론 명확성, 차별성 기준으로 평가하고,',
        'core / supporting / background / exclude로 다시 분류해줘.',
        '단, abstract 기반 평가는 예비 평가이며 full-text 확인 필요 항목을 별도로 표시해줘.',
        '```',
    ]
    return '\n'.join(lines)


md = to_markdown(top)
out = DATA_DIR / f'papers_top_{TOP_N}.md'
out.write_text(md, encoding='utf-8')
print(f'Saved -> {out.resolve()}')
print(md[:1600])
